# Chapter 12 — Entropy: modeling uncertainty

Companion notebook to `docs/models/ch12_entropy.md` and the Streamlit page `app/pages/12_Entropy.py`.

Three threads:
1. **Shannon entropy** — one number for uncertainty (and diversity).
2. **Maximum-entropy distributions** — the least-presumptuous guess given a constraint.
3. **Wolfram's four classes** on 1-D elementary cellular automata: equilibrium, periodic, random, complex.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import math

import matplotlib.pyplot as plt
import numpy as np

from modelthinker.dynamics.entropy import (
    WOLFRAM_CLASSES,
    exponential_entropy,
    normal_entropy,
    run_elementary_ca,
    shannon_entropy,
    uniform_entropy,
)

## 1. Shannon entropy — a warm-up

$H(p) = -\sum_i p_i \log_2 p_i$. Three sanity checks:
- A **certain** outcome has $H = 0$ bits.
- A **fair coin** has $H = 1$ bit.
- A **uniform** distribution over $K$ outcomes has $H = \log_2 K$.

In [ ]:
print(f"H(certain)      = {shannon_entropy([1, 0, 0, 0]):.3f} bits")
print(f"H(fair coin)    = {shannon_entropy([0.5, 0.5]):.3f} bits")
print(f"H(uniform-256)  = {shannon_entropy([1]*256):.3f} bits  (= log2(256) = 8)")

# Entropy as a *diversity* score for a market with 4 brands
shares = np.array([0.40, 0.30, 0.20, 0.10])
print(f"\nMarket diversity H = {shannon_entropy(shares):.3f} bits")
print(f"Maximum diversity  = log2(4) = {math.log2(4):.3f} bits (all brands equal)")

**Binary entropy curve.** For a 2-outcome distribution $(p, 1-p)$, entropy peaks at $p = 0.5$ and falls to 0 at either extreme. This is the shape of every information-theoretic story.

In [ ]:
ps = np.linspace(0, 1, 501)
H = np.array([shannon_entropy([p, 1 - p]) if 0 < p < 1 else 0.0 for p in ps])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ps, H, color='#4C6EF5', lw=2)
ax.axhline(1.0, color='#F76707', ls='--', lw=1, label='max = 1 bit')
ax.set(xlabel='p (probability of outcome 1)', ylabel='H (bits)',
       title='Binary entropy curve')
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 2. Maximum-entropy distributions

Given some constraint, the distribution with the **highest** entropy is the least-structured — it doesn't smuggle in unstated assumptions.

| Constraint | Max-entropy answer | Entropy (nats) |
|---|---|---|
| Known range $[a, b]$ | Uniform | $\ln(b - a)$ |
| Known mean $\mu > 0$ | Exponential | $1 + \ln \mu$ |
| Known mean & variance $\sigma^2$ | Normal | $\tfrac12 \ln(2\pi e \sigma^2)$ |

The `dynamics.entropy` module ships these closed-form entropies. Let's check them:

In [ ]:
print(f"H(Uniform(0, 8))     = {uniform_entropy(0, 8):.3f} bits  (= log2 8)")
print(f"H(Exp(mean = 1))     = {exponential_entropy(1.0):.3f} bits")
print(f"H(Normal(sigma = 1)) = {normal_entropy(1.0):.3f} bits")

# Notice: doubling variance adds 1/2 * log2(4) = 1 bit to the normal's entropy
print(f"\nH(Normal(sigma = 2)) = {normal_entropy(2.0):.3f} bits  (1 bit more than sigma=1)")

**Any structure you add costs entropy.** A bimodal distribution with the *same* variance as a normal has strictly less entropy — the extra clumping is information you claimed to know. Let's verify numerically on a discrete grid:

In [ ]:
xs = np.linspace(-6, 6, 601)
dx = xs[1] - xs[0]
sigma = 1.0
# Reference: a standard normal
normal_pdf = np.exp(-xs**2 / 2) / math.sqrt(2 * math.pi)
# A two-component mixture with total variance = 1 (two peaks at +/- d with inner sd s, d^2 + s^2 = 1)
d = 0.8; s_inner = math.sqrt(1 - d*d)
mix_pdf = 0.5 * (np.exp(-((xs - d)**2) / (2 * s_inner**2)) / (s_inner * math.sqrt(2 * math.pi))
                 + np.exp(-((xs + d)**2) / (2 * s_inner**2)) / (s_inner * math.sqrt(2 * math.pi)))

H_normal_disc = shannon_entropy(normal_pdf * dx + 1e-30)
H_mix_disc    = shannon_entropy(mix_pdf * dx + 1e-30)
print(f"Discrete H(normal, sigma=1)    = {H_normal_disc:.4f} bits")
print(f"Discrete H(bimodal, var=1)     = {H_mix_disc:.4f} bits")
print(f"Difference (structure penalty) = {H_normal_disc - H_mix_disc:.4f} bits")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(xs, normal_pdf, color='#4C6EF5', lw=2.5, label=f'Normal(0, 1) — H = {H_normal_disc:.3f}')
ax.plot(xs, mix_pdf,    color='#F76707', lw=2.5, label=f'Bimodal, var = 1 — H = {H_mix_disc:.3f}')
ax.set(xlabel='x', ylabel='density', title='Same variance, less entropy for the bimodal')
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 3. Wolfram's four classes — 1-D cellular automata

An **elementary CA** has a 1-D row of 0/1 cells and a rule that updates each cell from itself and its two neighbours. There are $2^{2^3} = 256$ possible rules. Wolfram noticed they all fall into four classes.

Below: four representatives, all starting from a single 1 at the centre.

In [ ]:
runs = {
    'Rule 0 · class 1 (equilibrium)': 0,
    'Rule 184 · class 2 (periodic — traffic)': 184,
    'Rule 30 · class 3 (random)':   30,
    'Rule 110 · class 4 (complex)': 110,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (title, rule) in zip(axes, runs.items(), strict=True):
    r = run_elementary_ca(rule=rule, width=201, steps=200, initial_kind='single')
    ax.imshow(r.grid, cmap='Greys', vmin=0, vmax=1, aspect='auto')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### Entropy signatures for each class

Row entropy = the 1-bit entropy of a single time-slice's 0/1 mix. Block entropy = entropy over length-$k$ horizontal windows over the tail (transients removed). The two together sort the four classes cleanly.

In [ ]:
rows = []
for name, rule in [('rule 0 (class 1)', 0), ('rule 184 (class 2)', 184),
                    ('rule 30 (class 3)', 30), ('rule 110 (class 4)', 110)]:
    r = run_elementary_ca(rule=rule, width=201, steps=200, initial_kind='single')
    rows.append((name, r.row_entropies.mean(), r.block_entropy_final, r.wolfram_class))

print(f'{"Rule":<22}  {"mean row H":>12}  {"block H (k=4)":>14}  {"detected class":>16}')
print('-' * 70)
for name, mean_row_h, block_h, cls in rows:
    print(f'{name:<22}  {mean_row_h:>12.4f}  {block_h:>14.4f}  {cls:>16}')

### Entropy over time — visualising the four regimes

Track row entropy step-by-step for the four representatives. Class 1 → drops to 0. Class 2 → oscillates in a narrow band. Class 3 → hugs the ceiling. Class 4 → fluctuates, structured but non-trivial.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, rule, color in [
    ('rule 0 (class 1)',    0,   '#868E96'),
    ('rule 184 (class 2)',  184, '#37B24D'),
    ('rule 30 (class 3)',   30,  '#E03131'),
    ('rule 110 (class 4)',  110, '#4C6EF5'),
]:
    r = run_elementary_ca(rule=rule, width=201, steps=200, initial_kind='single')
    ax.plot(r.row_entropies, label=name, color=color, lw=1.7)

ax.axhline(1.0, color='k', ls=':', lw=0.8, alpha=0.4)
ax.set(xlabel='time step', ylabel='row H (bits)',
       title='Row entropy over time for one rule per class', ylim=(-0.03, 1.05))
ax.legend(loc='center right'); ax.grid(alpha=0.3); plt.show()

### How well does our heuristic classifier match Wolfram's?

The `classify_ca` heuristic uses:
1. Tail all-zero or all-one → class 1.
2. Small periodicity in the tail → class 2.
3. Block entropy > 85% of max → class 3.
4. Everything else → class 4.

Compare that against the hand-classifications in `WOLFRAM_CLASSES`:

In [ ]:
matches = disagreements = 0
disagreement_examples = []
for rule, known in sorted(WOLFRAM_CLASSES.items()):
    r = run_elementary_ca(rule=rule, width=201, steps=200, initial_kind='single')
    if r.wolfram_class == known:
        matches += 1
    else:
        disagreements += 1
        if len(disagreement_examples) < 6:
            disagreement_examples.append((rule, known, r.wolfram_class, r.block_entropy_final))

total = matches + disagreements
print(f'Agreement on the well-known rules: {matches}/{total} = {100 * matches / total:.0f}%')
print('\nA few disagreements (rule · Wolfram class · heuristic class · block-H):')
for rule, known, got, bh in disagreement_examples:
    print(f'  rule {rule:3d}   {known} → {got}   (block H = {bh:.3f})')
print('\nInterpretation: the heuristic is easily fooled between class 3 and 4. Distinguishing')
print('rule 30 (random) from rule 110 (complex) by statistics alone is genuinely hard — that')
print('mismatch is precisely why Wolfram spent years on hand-classification.')

## 4. What-ifs to try

1. **Signal vs noise.** Pick a rule that produces a mix of order and chaos (try 54, 106, 137). Compute its row-entropy time-series. Where does it sit vs. the four representatives above?
2. **Initial-condition sensitivity.** Run rule 110 with `initial_kind='single'` vs `'random'`. Compare the space-time diagrams and the detected class. Which better fits the class-4 description, and why?
3. **Max-entropy check via sampling.** Draw 10,000 samples from `Exponential(mean = 2)`, discretise into 50 bins, and compute the empirical Shannon entropy. Compare to `exponential_entropy(2.0)`.
4. **Non-normal, same variance.** Cook up a distribution with variance = 1 but with lower entropy than `Normal(0, 1)`. What does your distribution's structure look like — where is the 'stolen' entropy hiding?